# Capitulo 14 - Carregamento de Dados e Chunking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap14_chunking.ipynb)

Neste capitulo, abordamos como carregar dados de diversas fontes e prepara-los
para indexacao em bancos vetoriais atraves de tecnicas de chunking (fragmentacao).

---

## Atribuicao e Fonte Original

**Fonte:** [RAG-with-Python-Cookbook](https://github.com/PacktPublishing/RAG-with-Python-Cookbook) - Packt Publishing

**Notebooks fonte:** ch03_loading_data/loading_data_to_RAG.ipynb, ch04_data_preparation_chunking_data/chunking_data.ipynb

**Adaptado e comentado por Allan Eric Jepsen** para o livro *LLM On-Premise: Guia Pratico*.

Todos os creditos aos autores originais sao mantidos.

---

## Instalacao de Dependencias

Execute a celula abaixo para instalar todos os pacotes necessarios.

In [ ]:
# Instalar dependencias do capitulo
%pip install -q python-docx python-magic-bin PyPDF2 pandas pydantic \
    tiktoken openai chromadb sentence-transformers python-dotenv requests

---
## Parte 1: Carregamento de Dados

Projetamos pipelines de carregamento para diferentes tipos de arquivo: PDF, DOCX, TXT, CSV.
Um bom pipeline de ingestao e fundamental para a qualidade do RAG.

## Pre-requisitos

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [ ]:
!
!pip install python-docx==1.1.2
!pip install python-magic-bin==0.4.14
!pip install pandas==2.2.3
!pip install PyPDF2==3.0.1
!pip install pillow==11.2.1
!pip install openpyxl==3.1.5
!pip install pdf2image==1.17.0
!pip install pytesseract==0.3.13
!pip install openai==1.82.1
!pip install python-dotenv==1.1.0
!pip install sqlalchemy==2.0.41
!pip install psycopg2-binary==2.9.10
!pip install moviepy==2.2.1
!pip install pdfminer.six==20250506
!pip install pi-heif==0.22.0
!pip install unstructured-inference==1.0.2
!pip install unstructured==0.17.2
!pip install -q "unstructured[pdf]" unstructured_pytesseract

### Install Poppler

To use `pdf2image` you need to install `Poppler`.

**Installation instructions:**
- **Google Colab / Ubuntu / Debian:** Run the cell below
- **macOS:** Run `brew install poppler` in your terminal
- **Windows:** Download from [poppler-windows releases](https://github.com/oschwartz10612/poppler-windows/releases/) and add to PATH

In [ ]:
import sys

# Verificar if running in Google Colab or Linux environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !apt-get update -qq
    !apt-get install -y -qq poppler-utils tesseract-ocr
    print("✓ Poppler and Tesseract-OCR installed successfully")
else:
    print("⚠ Running locally. Please install Poppler and Tesseract-OCR manually:")
    print("  - macOS: brew install poppler tesseract")
    print("  - Ubuntu/Debian: sudo apt-get install poppler-utils tesseract-ocr")
    print("  - Windows: Download Poppler from https://github.com/oschwartz10612/poppler-windows/releases/ and Tesseract-OCR from https://tesseract-ocr.github.io/tessdoc/Installation.html")

### Carregar arquivos de exemplo

This notebook uses sample Word and PDF files from the `datasets` directory.

When running on Google Colab, the code below will download the datasets from GitHub.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("✓ Datasets downloaded to /content/datasets")
else:
    print("⚠ Running locally. Using ../datasets/ directory")

### Carregar credenciais

If you run this code in Google Colab, save your OpenAI API key in the secrets and access it by

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Verificar if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

### 1.1 Loading Word Files in Python

Option 1: load word files using the `python_docx` library

In [ ]:
import os
import requests
from docx import Document
from io import BytesIO

file_path = "../datasets/word_files/2023_Jan_7_Feature_Engineering_Techniques.docx"
doc = Document(file_path)

text = []
for paragraph in doc.paragraphs:
    text.append(paragraph.text)

full_text = "\n".join(text)


In [ ]:
full_text

Option 2: load word files using the unstructured library

In [ ]:
from unstructured.partition.docx import partition_docx
import os
import pandas as pd

# elements = partition_docx(filename=file_path)
elements = partition_docx(filename=file_path)

list_of_elements = []

for element in elements:
    element_dict = {
        "element_id": element.id,
        "file_path": file_path,
        "category": element.category,  # e.g. "Title", "NarrativeText", "ListItem"
        "text": element.text,
        "last_modified": element.metadata.last_modified,
    }

    list_of_elements.append(element_dict)

elements_df = pd.DataFrame(list_of_elements)

In [ ]:
elements_df.head()

### 1.2 Loading PDF Files

In [ ]:
import PyPDF2
import os
import pandas as pd

file_path = "../datasets/pdf_files/2023_Jan_7_Feature_Engineering_Techniques.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    # Inicializar an empty string to store the extracted text
    list_of_pages = []
    page_counter = 1

    for page in reader.pages:
        page_dict = {
            "file_name": reader.metadata.get("/Title"),
            "producer": reader.metadata.get("/Producer"),
            "page_number": page_counter,
            "text": page.extract_text(),
            "images": page.images,
        }

        list_of_pages.append(page_dict)

        page_counter += 1

# Converter the list of pages to a pandas DataFrame
pages_df = pd.DataFrame(list_of_pages)

In [ ]:
# Exibir the first few rows of the DataFrame
pages_df.head()

### 1.3 Loading and Handling CSV and Excel Files

In [ ]:
import os
import pandas as pd

file_path = "../datasets/csv_files/census-income.xlsx"
df_excel = pd.read_excel(io=file_path)


def create_text_description_of_row(row):
    row["text_description"] = (
        f"""The candidate {row['age']} years old is working in the
            {row['workclass']} sector. The candidate was born in
            {row['native-country']}, is {row['marital-status']}
            and has a {row['relationship']} relationship.
            The candidate has a {row['education']} degree
            and is working as a {row['occupation']}.
            The income of the candidate is {row['income']}."""
    )

    return row


# Apply the function create_text_description_of_row to each row of the data frame
df_extended = df_excel.apply(create_text_description_of_row, axis=1)


In [ ]:
# Exibir the first 5 text_description of the dataset
df_extended["text_description"].head()

In [ ]:
df_extended["text_description"][0]

### 1.4 Querying a PostgreSQL Database

This section needs a PostgreSQL database to connect to.

**Note:** Make sure a PostgreSQL server is installed and running, either locally on your machine or on a remote host that you can connect to.

```
CREATE USER rag_user WITH PASSWORD 'raguserpassword123';
GRANT ALL ON ALL TABLES IN SCHEMA public TO rag_user;
```

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from psycopg2 import OperationalError

# PostgreSQL connection parameters
username = "rag_user"
password = "raguserpassword123"
host = "localhost"
port = "5432"
database = "postgres"

connection_string = (
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

try:
    engine = create_engine(connection_string)
    with engine.connect() as connection:
        query = """SELECT * FROM categories ORDER BY category_id ASC """
        result = pd.read_sql(query, connection)
        print(result)
except OperationalError as e:
    print(f"Error connecting to PostgreSQL database: {e}")
    print("Please ensure the PostgreSQL server is running and accessible.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### 1.5 Loading Audio Files by Using Speech-to-Text Models

In [ ]:
import os
from openai import OpenAI

audio_file_path = "../datasets/audio_files/harvard.wav"

# initialize the OpenAI client with your API key
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

with open(audio_file_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1", file=audio_file
    )

transcription

### 1.6 Extracting Text from Images and PDFs Using OCR

In [ ]:
import os
from pdf2image import convert_from_path
from PIL import Image
import pytesseract

image = Image.open(
    fp="../datasets/images/example_finance_reporting_slide.png"
)

text = pytesseract.image_to_string(image)
text

**Carregamento de dados:** Carregamos e processamos os documentos.

In [ ]:
import os
from pdf2image import convert_from_path
from PIL import Image
import pytesseract

file_path = (
    "../datasets/pdf_files/2023_Jan_7_Feature_Engineering_Techniques.pdf"
)

images = convert_from_path(pdf_path=file_path)

text = []
for i, image in enumerate(images):
    page_text = pytesseract.image_to_string(image)
    text.append(page_text)

print(text)

### 1.7 Extracting Text from Images using Multimodal Models

In [ ]:
import os
import base64
from openai import OpenAI

png_file_path = "../datasets/images/example_finance_reporting_slide.png"

# initialize the OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

with open(png_file_path, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    prompt = (
        "Extract the text from the image attached. Make sure to only "
        "extract only the text. If there is no text in the image, "
        "please return with the sentence 'No text found in the image."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # define the model to use
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                f"data:image/jpeg;base64,"
                                f"{base64_image}"
                            ),
                        },
                    },
                ],
            }
        ],
        max_completion_tokens=500,
    )

    content = response.choices[0].message.content
    print(content)



```
# This is formatted as code
```

### 1.8 Generating Text Summaries for Images Using Multimodal Models

In [ ]:
import os
import base64
from openai import OpenAI

image_path = "../datasets/images/vietnam.png"

# initialize the OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

with open(image_path, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")

    prompt = (
        "You are an assistant for visually impaired users. "
        "Describe the image in detail."
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": (
                                f"data:image/jpeg;base64,"
                                f"{base64_image}"
                            ),
                        },
                    },
                ],
            }
        ],
        max_completion_tokens=150,
    )

    content = response.choices[0].message.content
    print(content)

### 1.9 Generating Text Summaries for Embedded Tables Using Multimodal Models

The following code will use OCR and by default will use `unstructured_pytesseract`. Run the following to install `tessearct-ocr`.

In [ ]:
import os
from unstructured.partition.pdf import partition_pdf

pdf_file_path = "../datasets/pdf_files/adult_data_article.pdf"

tables = []
texts = []

# partition the PDF file into its elements
raw_pdf_elements = partition_pdf(
    filename=pdf_file_path,
    strategy="hi_res",
)

for element in raw_pdf_elements:
    if "unstructured.documents.elements.Table" in str(type(element)):
        tables.append(str(element))

**Configuracao de credenciais:** Configuramos as variaveis de ambiente e chaves de API.

In [ ]:
from openai import OpenAI
import pandas as pd


def summarize_tables(row):
    summary_prompt = (
        f"You are an assistant tasked with summarizing tables. "
        f"Give a concise summary of the table. "
        f"Table chunk: {row.table}"
    )

    # Inicializar the OpenAI API client and generate table summary
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": summary_prompt}],
        temperature=0.7,
        max_tokens=150,
    )

    row["table_summary"] = response.choices[0].message.content

    return row


# create a pandas dataframe from the tables
tables_df = pd.DataFrame(tables, columns=["table"])

# add a column to the dataframe to store the summaries
tables_df = tables_df.apply(summarize_tables, axis=1)

**Configuracao de credenciais:** Configuramos as variaveis de ambiente e chaves de API.

In [ ]:
# define a random question to the embedded table
user_question = "What are the education levels of the people working in Sales?"


def build_prompt_and_generate_answer(user_question, found_table):
    """
    This function builds a prompt using the user's question and the context of the table
    and generates an answer using the OpenAI API

    Parameters:
        user_question: the question asked by the user
        found_table: the table context to generate the answer from

    Returns:
        answered_question: the answer to the user's question
    """

    question_prompt = f"""You are an assistant using the content from PDFs \
                        to answer questions. Below you can find the \
                        user's question and relevant context. Please use the \
                        context to generate an answer to the user's question.

                        # User question: {user_question}

                        # Context:

                        ## Table summary:
                        {found_table.table_summary}

                        ## Table content:
                        {found_table.table}""".strip()

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    answered_question = (
        client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": question_prompt}],
            temperature=0.7,
            max_tokens=150,
        )
        .choices[0]
        .message.content
    )

    return answered_question


# generate the answer to the user's question
# as context we using the first entry in the tables_df
answered_question = build_prompt_and_generate_answer(
    user_question=user_question, found_table=tables_df.iloc[0]
)

print(answered_question)

### 1.10 Parsing PDFs with Multiple Media Content Using Unstructured and Multimodal Models

In [ ]:
from unstructured.partition.pdf import partition_pdf
import os

# set the OCR agent to tesseract
os.environ["OCR_AGENT"] = "tesseract"

pdf_file_path = "../datasets/pdf_files/adult_data_article.pdf"
image_output_dir = "../datasets/extracted_content_from_pdfs/images"

# create output directory if it doesn't exist
os.makedirs(image_output_dir, exist_ok=True)

# get elements using the function extract_pdf_elements
raw_pdf_elements = partition_pdf(
    filename=pdf_file_path,
    extract_images_in_pdf=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=False,
    extract_image_block_output_dir=image_output_dir,
)

# categorize elements by type
tables = []
texts = []
titles = []

# fill the just created lists with the elements
for element in raw_pdf_elements:
    element_type = str(type(element))
    if "unstructured.documents.elements.Table" in element_type:
        tables.append(str(element))
    elif "unstructured.documents.elements.NarrativeText" in element_type:
        texts.append(str(element))
    elif "unstructured.documents.elements.Title" in element_type:
        titles.append(str(element))

### 1.11 Loading Videos Using Speech-to-Text and Multimodal Models

You can find the test video used in this example on YouTube:  
[Learn Data Science Tutorial – Full Course for Beginners](https://www.youtube.com/watch?v=ua-CiDNNj30)

> **Note:** The MP4 file is not included in this repository due to its size.

In [ ]:
import os
import pandas as pd

from moviepy import VideoFileClip, TextClip, CompositeVideoClip

video_file_path = "../datasets/videos/learn-data-science-tutorial.mp4"
image_output_folder = "../datasets/videos/video_extracted_images"

# create output folder if it doesn't exist
os.makedirs(image_output_folder, exist_ok=True)

# Verificar if the video file exists
if os.path.exists(video_file_path):
    clip = VideoFileClip(video_file_path)

    # create a list of timestamps from which to extract a frame
    time_step = 10  # time in seconds
    timestamps = list(range(0, int(clip.duration) - time_step, time_step))

    # for each timestamp extract a frame
    for timestamp in timestamps:
        frame_image_path = os.path.join(
            image_output_folder, f"frame_{timestamp}.png"
        )
        clip.save_frame(frame_image_path, t=timestamp)
else:
    print(f"Video file not found: {video_file_path}. Skipping video processing.")
    timestamps = [] # Ensure timestamps is defined even if file not found

In [ ]:
# for each timestamp extract the audio sequence and save it to a .mp3 file
audio_output_folder = "../datasets/videos/video_extracted_audio"

# create output folder if it doesn't exist
os.makedirs(audio_output_folder, exist_ok=True)

for timestamp in timestamps:
    audio_clip = clip.subclip(timestamp, timestamp + time_step).audio
    output_audio_path = os.path.join(
        audio_output_folder, f"audio_{timestamp}.mp3"
    )
    audio_clip.write_audiofile(output_audio_path)

**Configuracao de credenciais:** Configuramos as variaveis de ambiente e chaves de API.

In [ ]:
import os
from openai import OpenAI


def audio_to_text(audio_path):
    """
    Convert audio to text using OpenAI's Whisper model.
    """

    if not os.path.exists(audio_path):
        print(f"[ERROR] File does not exist: {audio_path}")
        return None

    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    try:
        with open(audio_path, "rb") as audio_file:
            transcription = client.audio.transcriptions.create(
                model="whisper-1",
                file=audio_file
            )

        text_file_path = os.path.splitext(audio_path)[0] + ".txt"

        with open(text_file_path, "w", encoding="utf-8") as text_file:
            text_file.write(transcription.text)

        return transcription.text

    except FileNotFoundError:
        print(f"[ERROR] File not found during processing: {audio_path}")
    except PermissionError:
        print(f"[ERROR] Permission denied: {audio_path}")
    except Exception as e:
        print(f"[ERROR] Failed to process {audio_path}: {e}")

    return None


# Processar folder
audio_files = os.listdir(audio_output_folder)

for audio_file in audio_files:
    absolute_path_audio_file = os.path.join(audio_output_folder, audio_file)

    # No need to check exists here anymore, function handles it
    result = audio_to_text(audio_path=absolute_path_audio_file)

    if result is None:
        print(f"Skipping file: {absolute_path_audio_file}")

---
## Parte 2: Estrategias de Chunking

Chunking e o processo de dividir documentos longos em fragmentos menores.
A estrategia de chunking impacta diretamente a qualidade da recuperacao.
Exploramos: chunking por tamanho fixo, por sentencas, semantico e recursivo.

## Pre-requisitos

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [ ]:
!pip install PyPDF2==3.0.1
!pip install pandas==2.2.3
!pip install pydantic==2.11.5
!pip install openai==1.83.0
!pip install matplotlib==3.10.3
!pip install scikit-learn==1.6.1
!pip install python-docx==1.1.2
!pip install nltk==3.9.1
!pip install langchain==0.3.25
!pip install langchain_openai==0.3.21
!pip install langchain-experimental==0.3.4
!pip install python-dotenv==1.0.0

### Setting Up Sample Files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

### Setting Up API Secrets

If you run this code in Google Colab, save your OpenAI API key in the colab secrets and load it to the environmental variables.

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Verificar if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

## Metadata Extraction and Filtering

In [ ]:
import PyPDF2
import os

file_path = "../datasets/pdf_files/attention_is_all_you_need_paper.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    metadata = reader.metadata

    text = ""
    for page in reader.pages:
        text += page.extract_text()
metadata


In [ ]:
metadata_ext = dict(metadata)
metadata_ext["page_count"] = len(reader.pages)
metadata_ext["file_size"] = os.path.getsize(file_path)
metadata_ext["file_name"] = os.path.basename(file_path)
metadata_ext["file_path"] = file_path
metadata_ext["text_length"] = len(text)
metadata_ext


**Geracao LLM:** Utilizamos o modelo de linguagem para gerar resposta.

In [ ]:
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI()

class AuthorContact(BaseModel):
    name: str
    company: str
    email: list[str]

class Contacts(BaseModel):
    entries: list[AuthorContact]

system_message = """Extract the contact information of all authors."""

response = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[
        {
            "role": "system",
            "content": system_message,
        },
        {
            "role": "user",
            "content": text,
        },
    ],
    response_format=Contacts,
)

author_contacts = response.choices[0].message.parsed

metadata_ext_llm = dict(metadata_ext)
metadata_ext_llm["author_contacts"] = author_contacts
metadata_ext_llm


## Data Quality Enhancement: Abbreviation Expansion

In [ ]:
import re

abbreviations_dict = {
    "NLP": "Natural Language Processing",
    "RNN": "Recurrent Neural Network",
    "LSTM": "Long Short-Term Memory",
    "GRU": "Gated Recurrent Unit",
    "TF": "Transformer",
    "MHA": "Multi-Head Attention",
    "FFN": "Feed-Forward Network",
}

file_path = "../datasets/text_files/blog_post_transformers.txt"
with open(file_path, "r") as file:
    text = file.read()

# Replace abbreviations in the text
for abbr, full_form in abbreviations_dict.items():
    text = re.sub(rf"\b{abbr}\b", f"{full_form} ({abbr})", text)

text


**Geracao LLM:** Utilizamos o modelo de linguagem para gerar resposta.

In [ ]:
import os
from openai import OpenAI

file_path = "../datasets/text_files/EMEA_drives_revenue.txt"

with open(file_path, "r") as file:
    text = file.read()

prompt = f"""
    The text below contains a financial report including a lot of
    abbreviations and technical terms from the finance domain.
    Please replace the abbreviations with their full forms and
    provide a brief explanation of the technical terms, so the
    whole text gets easier to read and understandable for everyone.

    Make sure it's easy enough, so that a 10-year-old school kid could
    understand it.

    Often used abbreviations are:
    - EMEA: Europe, Middle East, and Africa
    - BD: Business Development
    - YoY: Year-over-Year
    - APAC: Asia-Pacific

    Text:
    {text}
    """.strip()

client = OpenAI()
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="gpt-5.2",
)

enhanced_text = chat_completion.choices[0].message.content
enhanced_text


In [ ]:
# write enhanced_text to a new .txt file
output_file_path = "../datasets/text_files/EMEA_drives_revenue_enhanced.txt"
with open(output_file_path, "w") as file:
    file.write(enhanced_text)

## Hypothetical Question Embedding for Enhanced Retrieval

In [ ]:
import PyPDF2

file_path = "../datasets/pdf_files/AI_in_Factories_Discussion_Cleaned.pdf"

with open(file_path, "rb") as file:
    # Criar PDF reader object
    reader = PyPDF2.PdfReader(file)

    # Extrair text from all pages
    text = ""
    for page in reader.pages:
        text += page.extract_text()

text


**Chunking:** Dividimos o texto em fragmentos menores.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n",
    ],
)

text_chunks = text_splitter.split_text(text)
text_chunks

**Geracao LLM:** Utilizamos o modelo de linguagem para gerar resposta.

In [ ]:
import textwrap
from openai import OpenAI
from pydantic import BaseModel

file_path = "../datasets/text_files/AI_in_factories_chat.txt"

with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

client = OpenAI()

prompt = textwrap.dedent(
    f"""
    Below you can find a chat history between two students.

    Please generate 5 hypothetical questions that could be
    answered using the information from the discussion.
    The questions should focus on key details, definitions, and
    information present in the text.

    Chat History:
    {text}
    """
)

class HypotheticalQuestions(BaseModel):
    questions: list[str]

result = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": prompt}],
    response_format=HypotheticalQuestions,
)

hypothetical_questions = result.choices[0].message.parsed.questions
hypothetical_questions


## Character-Based Document Splitting

In [ ]:
file_path = "../datasets/text_files/blog_post_transformers.txt"

with open(file_path, "r") as file:
    text = file.read()

def split_by_characters(text, chunk_size, overlap):
    chunks = []
    step = max(1, chunk_size - overlap)

    for start in range(0, len(text), step):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk:
            chunks.append(chunk)

    return chunks

chunks = split_by_characters(text, chunk_size=100, overlap=20)
chunks


## Recursive Text Splitting Methods

In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
import PyPDF2

file_path = "../datasets/pdf_files/daily_insights.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    text = ""
    for page in reader.pages:
        text += page.extract_text()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_text(text)
chunks


## Document-Aware Splitting

In [ ]:

import os
from langchain_text_splitters import (
    PythonCodeTextSplitter,
    LatexTextSplitter,
    MarkdownHeaderTextSplitter,
)

file_path = "../datasets/markdown_files/random_md_code.md"
file_extension = os.path.splitext(file_path)[1]

with open(file_path, "r") as file:
    file_text = file.read()

if file_extension == ".py":
    splitter = PythonCodeTextSplitter(chunk_size=500, chunk_overlap=50)
elif file_extension == ".tex":
    splitter = LatexTextSplitter(chunk_size=500, chunk_overlap=50)
elif file_extension == ".md":
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

chunks = splitter.split_text(file_text)
chunks

## Semantic-Aware Text Chunking

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

file_path = (
    "../datasets/text_files/"
    "random-text-about-5-different-stories.txt"
)

with open(file_path, "r") as file:
    text = file.read()

text_splitter = SemanticChunker(
    OpenAIEmbeddings(),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)

chunks = text_splitter.split_text(text)
chunks


## Agentic Text Chunking

In [ ]:
from langchain import hub
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List

# pull the prompt template from the langchain hub
obj = hub.pull("wfh/proposal-indexing")

llm = ChatOpenAI(model="gpt-5.2")

class Sentences(BaseModel):
    sentences: List[str]

extraction_llm = llm.with_structured_output(Sentences)

# Criar the sentence extraction chain
extraction_chain = obj | extraction_llm

propositions = extraction_chain.invoke(
    """
    On July 20, 1969, astronaut Neil Armstrong walked on the moon .
    He was leading the NASA's Apollo 11 mission.
    Armstrong famously said, "That's one small step for man, one
    giant leap for mankind" as he stepped onto the lunar surface.
    """
)


In [ ]:
for sentence in propositions.sentences:
    print(sentence)